<a href="https://colab.research.google.com/github/ChandanGurjar/Exploratory-Data-Analysis-EDA-ML-Models/blob/main/NLP-Projects/Resume_Skill_Extraction_Named_Entity_Recognition.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

### Mounting Google drive


In [ ]:
from google.colab import drive
drive.mount('/content/drive')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [ ]:
%cd '/content/drive/My Drive/five_class_entitydata'

/content/drive/My Drive/five_class_entitydata


### Imports


In [ ]:
import os
import spacy
import re
import json
import random
import logging
from sklearn.metrics import classification_report
from sklearn.metrics import precision_recall_fscore_support
from spacy.training.gold_parse import GoldParse
from spacy.training.scorer import Scorer
from sklearn.metrics import accuracy_score # Removed since it's not used

ModuleNotFoundError: No module named 'spacy.training.gold_parse'

In [ ]:
import os
import spacy
import re
import json
import random
import logging
from sklearn.metrics import classification_report
from sklearn.metrics import precision_recall_fscore_support
from spacy.training import Example
from spacy.scorer import Scorer

In [ ]:
!pip install -U spacy[transformers]

### Converting dataturks to spacy format


In [ ]:
#converting dataturks annotated data to spacy format to be
#used as training data

def convert_dataturks_to_spacy(dataturks_JSON_FilePath):
    try:
        training_data = []
        lines=[]
        with open(dataturks_JSON_FilePath, 'r') as f:
            lines = f.readlines()

        for line in lines:
            data = json.loads(line)
            text = data['content']
            entities = []
            for annotation in data['annotation']:
                #only a single point in text annotation.
                point = annotation['points'][0]
                labels = annotation['label']
                # handle both list of labels or a single label.
                if not isinstance(labels, list):
                    labels = [labels]

                for label in labels:
                    #dataturks indices are both inclusive [start, end] but spacy is not [start, end)
                    entities.append((point['start'], point['end'] + 1 ,label))


            training_data.append((text, {"entities" : entities}))

        return training_data
    except Exception as e:
        logging.exception("Unable to process " + dataturks_JSON_FilePath + "\n" + "error = " + str(e))
        return None

### Cleaning data


In [ ]:
############################Removes leading and trailing white spaces from entity spans.############################
# https://github.com/explosion/spaCy/issues/3558
def trim_entity_spans(data: list) -> list:
    """Removes leading and trailing white spaces from entity spans.

    Args:
        data (list): The data to be cleaned in spaCy JSON format.

    Returns:
        list: The cleaned data.
    """
    invalid_span_tokens = re.compile(r'\s')

    cleaned_data = []
    for text, annotations in data:
        entities = annotations['entities']
        valid_entities = []
        for start, end, label in entities:
            valid_start = start
            valid_end = end
            while valid_start < len(text) and invalid_span_tokens.match(
                    text[valid_start]):
                valid_start += 1
            while valid_end > 1 and invalid_span_tokens.match(
                    text[valid_end - 1]):
                valid_end -= 1
            valid_entities.append([valid_start, valid_end, label])
        cleaned_data.append([text, {'entities': valid_entities}])

    return cleaned_data

### Training the model


In [ ]:
################### Train Spacy NER.###########
def train_spacy():
    # !!! IMPORTANT: Correct the file path below to the actual path of your training data JSON file in Google Drive !!!
    TRAIN_DATA = convert_dataturks_to_spacy("/content/drive/MyDrive/five_class_entitydata/traindata_3withmyannotation.json")
    TRAIN_DATA=trim_entity_spans(TRAIN_DATA)
    nlp = spacy.blank('en')  # create blank Language class
    # create the built-in pipeline components and add them to the pipeline
    # nlp.create_pipe works for built-ins that are registered with spaCy
    # if 'tagger' not in nlp.pipe_names:
    #      nlp.add_pipe(nlp.create_pipe('tagger'))
    if 'ner' not in nlp.pipe_names:
        # Use the string name of the component factory with add_pipe
        ner = nlp.add_pipe('ner', last=True)


    # add labels
    for _, annotations in TRAIN_DATA:
         for ent in annotations.get('entities'):
            ner.add_label(ent[2])

    # get names of other pipes to disable them during training
    other_pipes = [pipe for pipe in nlp.pipe_names if pipe != 'ner']
    with nlp.disable_pipes(*other_pipes):  # only train NER
        optimizer = nlp.begin_training()
        for itn in range(25):
            print("Statring iteration " + str(itn))
            random.shuffle(TRAIN_DATA)
            losses = {}
            for text, annotations in TRAIN_DATA:
                nlp.update(
                    [text],  # batch of texts
                    [annotations],  # batch of annotations
                    drop=0.1,  # dropout - make it harder to memorise data
                    sgd=optimizer,  # callable to update weights
                    losses=losses)
            print(losses)
    return nlp

In [ ]:
import spacy
import random
from spacy.training import Example
from spacy.util import minibatch, compounding

# It's assumed your helper functions are defined elsewhere in your code:
# from your_data_loader_file import convert_dataturks_to_spacy, trim_entity_spans

# --- HELPER FUNCTION TO FIX OVERLAPS ---
def clean_overlapping_entities(training_data):
    """
    Removes overlapping entities from the training data.
    Keeps the first entity encountered in a sorted list.
    """
    cleaned_data = []
    for text, annotations in training_data:
        entities = annotations.get('entities', [])
        if not entities:
            cleaned_data.append((text, annotations))
            continue

        # Sort entities by their start character index
        entities = sorted(entities, key=lambda x: x[0])

        non_overlapping_entities = []
        if entities:
            # Add the first entity to our list of non-overlapping entities
            non_overlapping_entities.append(entities[0])
            last_end = entities[0][1]

            # Iterate through the rest of the entities
            for ent in entities[1:]:
                start, end, label = ent
                # If the current entity does not overlap with the last one added, keep it
                if start >= last_end:
                    non_overlapping_entities.append(ent)
                    last_end = end

        cleaned_annotations = {'entities': non_overlapping_entities}
        cleaned_data.append((text, cleaned_annotations))

    return cleaned_data


################### Train Spacy NER.###########
def train_spacy():
    # --- Step 1: Load and prepare your training data ---
    TRAIN_DATA = convert_dataturks_to_spacy("/content/drive/MyDrive/five_class_entitydata/traindata_3withmyannotation.json")
    TRAIN_DATA = trim_entity_spans(TRAIN_DATA)

    # --- Step 2: Clean the data to remove overlapping entities ---
    # This new line calls the helper function to fix the error
    TRAIN_DATA = clean_overlapping_entities(TRAIN_DATA)

    # It is highly recommended to start from a pretrained model for better results
    nlp = spacy.load('en_core_web_sm')

    ner = nlp.get_pipe('ner')

    # --- Step 3: Add your custom entity labels to the NER component ---
    for _, annotations in TRAIN_DATA:
          for ent in annotations.get('entities'):
            ner.add_label(ent[2])

    # --- Step 4: Convert your data into Example objects for training ---
    examples = []
    for text, annotations in TRAIN_DATA:
        doc = nlp.make_doc(text)
        examples.append(Example.from_dict(doc, annotations))

    # --- Step 5: Train the NER model ---
    other_pipes = [pipe for pipe in nlp.pipe_names if pipe != 'ner']
    with nlp.disable_pipes(*other_pipes):  # only train NER
        optimizer = nlp.begin_training()
        for itn in range(25): # Consider increasing iterations to 100 or more
            print("Starting iteration " + str(itn))
            random.shuffle(examples)
            losses = {}

            batches = minibatch(examples, size=compounding(4.0, 32.0, 1.001))
            for batch in batches:
                nlp.update(
                    batch,
                    drop=0.1,
                    sgd=optimizer,
                    losses=losses
                )
            print(losses)

    return nlp

# You can now run your training function:
# nlp_model = train_spacy()

In [ ]:
import spacy    # training the model for 100 iterations
import random
from spacy.training import Example
from spacy.util import minibatch, compounding

# It's assumed your helper functions are defined elsewhere in your code:
# from your_data_loader_file import convert_dataturks_to_spacy, trim_entity_spans

# --- HELPER FUNCTION TO FIX OVERLAPS ---
def clean_overlapping_entities(training_data):
    """
    Removes overlapping entities from the training data.
    Keeps the first entity encountered in a sorted list.
    """
    cleaned_data = []
    for text, annotations in training_data:
        entities = annotations.get('entities', [])
        if not entities:
            cleaned_data.append((text, annotations))
            continue

        # Sort entities by their start character index
        entities = sorted(entities, key=lambda x: x[0])

        non_overlapping_entities = []
        if entities:
            # Add the first entity to our list of non-overlapping entities
            non_overlapping_entities.append(entities[0])
            last_end = entities[0][1]

            # Iterate through the rest of the entities
            for ent in entities[1:]:
                start, end, label = ent
                # If the current entity does not overlap with the last one added, keep it
                if start >= last_end:
                    non_overlapping_entities.append(ent)
                    last_end = end

        cleaned_annotations = {'entities': non_overlapping_entities}
        cleaned_data.append((text, cleaned_annotations))

    return cleaned_data


################### Train Spacy NER.###########
def train_spacy():
    # --- Step 1: Load and prepare your training data ---
    TRAIN_DATA = convert_dataturks_to_spacy("/content/drive/MyDrive/five_class_entitydata/traindata_3withmyannotation.json")
    TRAIN_DATA = trim_entity_spans(TRAIN_DATA)

    # --- Step 2: Clean the data to remove overlapping entities ---
    TRAIN_DATA = clean_overlapping_entities(TRAIN_DATA)

    # It is highly recommended to start from a pretrained model for better results
    nlp = spacy.load('en_core_web_sm')

    ner = nlp.get_pipe('ner')

    # --- Step 3: Add your custom entity labels to the NER component ---
    for _, annotations in TRAIN_DATA:
          for ent in annotations.get('entities'):
            ner.add_label(ent[2])

    # --- Step 4: Convert your data into Example objects for training ---
    examples = []
    for text, annotations in TRAIN_DATA:
        doc = nlp.make_doc(text)
        examples.append(Example.from_dict(doc, annotations))

    # --- Step 5: Train the NER model ---
    other_pipes = [pipe for pipe in nlp.pipe_names if pipe != 'ner']
    with nlp.disable_pipes(*other_pipes):  # only train NER
        optimizer = nlp.begin_training()

        # Increased training duration from 25 to 100 iterations
        for itn in range(50):
            print("Starting iteration " + str(itn))
            random.shuffle(examples)
            losses = {}

            batches = minibatch(examples, size=compounding(4.0, 32.0, 1.001))
            for batch in batches:
                nlp.update(
                    batch,
                    drop=0.1,
                    sgd=optimizer,
                    losses=losses
                )
            print(losses)

    return nlp

# You can now run your training function:
# nlp_model = train_spacy()

In [ ]:
!pip install spacy-lookups-data


In [ ]:
nlp_=train_spacy()

Starting iteration 0
{'ner': np.float32(64029.043)}
Starting iteration 1
{'ner': np.float32(7015.557)}
Starting iteration 2
{'ner': np.float32(5075.1763)}
Starting iteration 3
{'ner': np.float32(5559.083)}
Starting iteration 4
{'ner': np.float32(4383.773)}
Starting iteration 5
{'ner': np.float32(3482.9895)}
Starting iteration 6
{'ner': np.float32(2893.6858)}
Starting iteration 7
{'ner': np.float32(2682.55)}
Starting iteration 8
{'ner': np.float32(2431.944)}
Starting iteration 9
{'ner': np.float32(2298.669)}
Starting iteration 10
{'ner': np.float32(1987.1112)}
Starting iteration 11
{'ner': np.float32(1861.8306)}
Starting iteration 12
{'ner': np.float32(1780.307)}
Starting iteration 13
{'ner': np.float32(1518.2523)}
Starting iteration 14
{'ner': np.float32(1459.9938)}
Starting iteration 15
{'ner': np.float32(1461.0413)}
Starting iteration 16
{'ner': np.float32(1617.4906)}
Starting iteration 17
{'ner': np.float32(1280.6152)}
Starting iteration 18
{'ner': np.float32(1009.5623)}
Starting it

### Saving the Trained model


In [ ]:
# save model to output directory (with parcial cleaned data)
def save_model(output_dir):
      nlp_.to_disk(output_dir)
      print("Saved model to", output_dir)


In [ ]:
output_dir='./model2'
save_model(output_dir)

Saved model to ./model2


### Loading the trained model instance


In [ ]:
 ###################loading the saved model################################
 output_dir='./model2'
 nlp2 = spacy.load(output_dir)

### Testing


In [ ]:
##############################preparing the testdata########################
examples = convert_dataturks_to_spacy("3class_test_data.json")
examples=trim_entity_spans(examples)
tp = 0
tr = 0
tf = 0

ta = 0
c = 0


In [ ]:
#################testing the model######################
nlp_=nlp2
for text, annot in examples:

    f = open("resume"+str(c)+".txt", "w")
    doc_to_test = nlp_(text)
    d = {}
    for ent in doc_to_test.ents:
        d[ent.label_] = []
    for ent in doc_to_test.ents:
        d[ent.label_].append(ent.text)

    if 'Skills' in d:
      skills_=d['Skills']
      print(f'resume {str(c)} skills {skills_}')
    # print(d.keys())

    #---------------------------
    for i in set(d.keys()):

        f.write("\n\n")
        f.write(i + ":"+"\n")
        for j in set(d[i]):
            f.write(j.replace('\n', '')+"\n")
    #-----------------------------
    d = {}
    for ent in doc_to_test.ents:
        d[ent.label_] = [0, 0, 0, 0, 0, 0]
    for ent in doc_to_test.ents:
        doc_gold_text = nlp_.make_doc(text)
        gold = GoldParse(doc_gold_text, entities=annot.get("entities"))
        y_true = [ent.label_ if ent.label_ in x else 'Not ' +
                  ent.label_ for x in gold.ner]
        y_pred = [x.ent_type_ if x.ent_type_ ==
                  ent.label_ else 'Not '+ent.label_ for x in doc_to_test]
        if(d[ent.label_][0] == 0):
            # f.write("For Entity "+ent.label_+"\n")
            # f.write(classification_report(y_true, y_pred)+"\n")
            (p, r, f, s) = precision_recall_fscore_support(
                y_true, y_pred, average='weighted')
            a = accuracy_score(y_true, y_pred)
            d[ent.label_][0] = 1
            d[ent.label_][1] += p
            d[ent.label_][2] += r
            d[ent.label_][3] += f
            d[ent.label_][4] += a
            d[ent.label_][5] += 1
    c += 1

resume 0 skills ['Salesforce']


NameError: name 'GoldParse' is not defined

In [ ]:
from spacy.training import Example
from spacy.scorer import Scorer
import logging

#################testing the model######################
# Assume 'nlp2' is your trained model and 'examples' is your test data
# in the format: [("text", {"entities": [...]}), ...]
nlp_ = nlp2
c = 0 # Your counter variable

# 1. Initialize the Scorer and a list to hold Example objects
scorer = Scorer()
examples_for_scoring = []

# This loop now does two things:
# A) Saves the predicted entities for each resume to a file.
# B) Collects "Example" objects for evaluation later.
for text, annot in examples:
    # --- Part A: Process and save results (your original code) ---
    f = open("resume"+str(c)+".txt", "w")
    pred_doc = nlp_(text) # This is the model's prediction

    d = {}
    for ent in pred_doc.ents:
        d.setdefault(ent.label_, []).append(ent.text)

    if 'Skills' in d:
        skills_ = d['Skills']
        print(f'resume {str(c)} skills {skills_}')

    for i in set(d.keys()):
        f.write("\n\n")
        f.write(i + ":"+"\n")
        for j in set(d[i]):
            f.write(j.replace('\n', '')+"\n")
    f.close() # Good practice to close the file

    # --- Part B: Create an Example object for scoring ---
    # This bundles the prediction (pred_doc) with the correct answer (annot)
    try:
        example = Example.from_dict(pred_doc, annot)
        examples_for_scoring.append(example)
    except ValueError as e:
        # Catch the specific error for overlapping entities
        if "[E103]" in str(e):
            logging.warning(f"Skipping example {c} due to overlapping entities in annotations: {e}")
        else:
            # Re-raise other ValueErrors
            raise e


    c += 1

# 2. Evaluate the model's performance on all examples at once
# This happens AFTER the loop has finished.
scores = scorer.score(examples_for_scoring)

# 3. Print the final results
print("\n------------------ OVERALL MODEL PERFORMANCE ------------------")
print(f"Precision: {scores['ents_p']:.4f}")
print(f"Recall: {scores['ents_r']:.4f}")
print(f"F1-Score: {scores['ents_f']:.4f}")
print("\n------------------ PERFORMANCE PER ENTITY ------------------")
for entity, metrics in scores['ents_per_type'].items():
    print(f"\nEntity: {entity}")
    print(f"  Precision: {metrics['p']:.4f}")
    print(f"  Recall: {metrics['r']:.4f}")
    print(f"  F1-Score: {metrics['f']:.4f}")

resume 0 skills ['IBM Watson']
resume 1 skills ['Machine learning', 'Python', 'Matlab', 'Python,', 'SQL', 'Splunk', 'Hadoop', 'ABAP', 'Web services', 'Java']


resume 2 skills ['Python', 'Python', 'Shell', 'Perl', 'C', 'C++', 'Hadoop', 'vectorwise', 'CSS', 'HTML', 'Django', 'CSS', 'Python', 'Python', 'jquery', 'CSS', 'Python', 'Python', 'Python', 'Apache', 'Jquery', 'Apache', 'Jquery', 'Python', 'Python', 'Java', 'Web Development']
resume 3 skills ['R', 'SAS', 'machine learning', 'R', 'SAS', 'R', 'SPSS']
resume 4 skills ['machine learning', 'Python', 'Python', 'Machine Learning', 'Big Data-Learning', 'HDFS', 'R']


resume 5 skills ['Devops', 'SQL', 'Machine Learning', 'R', 'DB2', 'R', 'JAVA', 'SQL', 'VB', 'Python', 'Java', 'DB2', 'SAP', 'IBM Global', 'R', 'Watson Analytics', 'SPSS', 'Python', 'SQL', 'DB2']


resume 6 skills ['R', 'Python', 'R', 'Python', 'Machine Learning', 'R', 'Semi Structured', 'SQL', 'SQL', 'R', 'Python', 'Java', 'Scala', 'HDFS,', 'Spark', 'Oracle', 'Python', 'Machine Learning', 'NLP', 'Text mining', 'MySql', 'R', 'Python', 'R', 'Python', 'Python', 'MySQL', 'Python', 'Machine Learning', 'Deep Learning', 'HDFS', 'R', 'Python', 'MySQL', 'WCF', 'AJAX', '4.5.1,C#.NET', 'ASP.NET', 'XML', 'JAVA SCRIPT', 'XML', 'JAVA SCRIPT', 'AJAX', 'SQL SERVER 2005\n     ']
resume 7 skills ['Machine learning', 'Java/J2EE', 'Python', 'Machine Learning', 'Python', 'Java']


resume 8 skills ['Predictive modelling', 'Statistical Modelling', 'Machine Learning', 'R', 'Python', 'MBA', 'R', 'Python', 'Python', 'SQL', 'Machine Learning', 'Python', 'Python', 'R', 'Python', 'SQL']


resume 9 skills ['Image Processing', 'Machine Learning', 'VC++', 'C++', 'Python', 'C++', 'Python', 'C++', 'C', 'C/C++', 'Machine Learning', 'Machine Learning', 'Python', 'Tesseract', 'OCR', 'Tesseract', 'C++', 'C#.net']
resume 10 skills ['Python', 'Javascript', 'CSS', 'CSS', 'Javascript', 'Python', 'Python', 'Javascript', 'CSS', 'Python 2.7', 'Python', 'Python']
resume 11 skills ['Deep Learning', 'Machine Learning', 'Artificial intelligence', 'Python/R', 'Deep Learning', 'BigData', 'Deep Learning', 'Python', 'Machine Learning', 'Python', 'Shell script', 'Python', 'Machine Learning', 'Shell script', 'Perl', 'Python']
resume 12 skills ['data mining', 'Java', 'JavaScript', 'R-programming', 'data mining', 'Lotus Notes', 'Java', 'Lotus Notes']


resume 13 skills ['R / Python /', 'Data Analytics', 'SAS', 'SAS', 'Python', 'R', 'Mainframe', 'COBOL', 'DB2', 'R Programing', 'Python', 'SQL', 'SAS', 'Tableau', 'Spark', 'Hive', 'Cobol', 'DB2']
resume 14 skills ['Python', 'SQL', 'R', 'Python']
resume 15 skills ['R', 'R', 'SQL', 'Tableau', 'SAS']
resume 16 skills ['Bangalore', 'Bangalore', 'Python', 'Javascript', 'Apache', 'Django', 'Python', 'JavaScript', 'JQuery', 'Apache', 'MySQL', 'Python', 'JavaScript', 'JQuery', 'Django', 'Apache', 'MySql', 'Python', 'JavaScript', 'JQuery', 'Apache', 'MySQL', 'PHP', 'JavaScript', 'JQuery']
resume 17 skills ['Python programming', 'Python', 'Python', 'Machine Learning(Linear', 'Python', 'Python', 'Python', 'Jenkins']
resume 18 skills ['R', 'Python', 'Tableau', 'Machine Learning', 'Tableau', 'R', 'Python', 'Python', 'Python']
resume 20 skills ['Python', 'Python', 'XML', 'C#', 'Python']
resume 21 skills ['IBM Watson', 'WKS-Watson', 'SQL', 'IBM Watson', 'SQL', 'SQL', 'Watson Content Analytics', 'Watson

### Validating the pridiction


In [ ]:
###########################validating the model##########################
for i in d:
    print("\n For Entity "+i+"\n")
    print("Accuracy : "+str((d[i][4]/d[i][5])*100)+"%")
    print("Precision : "+str(d[i][1]/d[i][5]))
    print("Recall : "+str(d[i][2]/d[i][5]))
    print("F-score : "+str(d[i][3]/d[i][5]))


 For Entity Skills



IndexError: list index out of range

### matcher


In [ ]:
import pandas as pd
from pathlib import Path

nlp_=nlp2

def find_skills(text):
  d = {}
  docx=nlp_(text)
  for ent in docx.ents:
    d[ent.label_] = []
  for ent in docx.ents:
    d[ent.label_].append(ent.text)
  if 'Skills' in d:
    skills_=d['Skills']
    return skills_
  else:
    return None

### Creating job list


In [ ]:
# create jobs list
jobs=[]
job_dir='/content/drive/My Drive/five_class_entitydata/jobs'
pathlist = Path(job_dir).glob('**/*.txt')
for path in pathlist:
    with open (path, "r") as fileHandler:
      job={
          'name':path.name,
           'skills':find_skills(''.join(fileHandler.readlines()))
      }
      jobs.append(job)



In [ ]:
print(jobs[1]['name'])
print(jobs[1]['skills'])
print(jobs[2]['name'])
print(jobs[2]['skills'])
print(jobs[3]['name'])
print(jobs[3]['skills'])
print(jobs[4]['name'])
print(jobs[4]['skills'])

javadeveloper.txt
['SQL Server', 'IHS', 'WAS', 'Java EE', 'SQL Server', '.NET core', 'C#', 'ASP.NET', 'Rdlc', 'Linq', 'Sql', 'Web Api', 'Mvc', 'Javascript', 'Web Services', 'Oracle', 'MS SQL']
phpdeveloper.txt
['PHP', 'Laravel', 'CodeIgniter', 'Symfony', 'Zend', 'Phalcon', 'CakePHP', 'Yii', 'FuelPHP', 'React', 'Vue', 'Angular', 'Ember', 'Backbone']
datascientist.txt
['Data Science', 'Python', 'Machine Learning', 'SAS', 'Java', 'Scala', 'Hadoop', 'Hive', 'Bigdata', 'Programming', 'SQL server reporting', 'Msbi Ssis', 'Ssrs', 'Msbi', 'Sql Reporting', 'Artificial Intelligence', 'Pandas', 'Pyspark', 'Sklearn', 'Flask', 'Django', 'Map Reduce', 'Parametric Design', 'Modeling', 'Regression', 'Patterns', 'Data Mining', 'Text Mining', 'Oops', 'Deep Learning', 'Web Analytics', 'Time Series', 'Regression', 'Tensorflow', 'Azure', 'Linear Regression', 'Logistic Regression', 'Decision Tree', 'Random Forest', 'Data Structure', 'Computer Vision']
dataengineer.txt
['J2EE', 'Oracle Fusion', 'Oracle Cloud

### Creating cv list


In [ ]:
# create cvs list
cvs=[]
cv_dir='/content/drive/My Drive/five_class_entitydata/cv'
pathlist = Path(cv_dir).glob('**/*.txt')
for path in pathlist:
    with open (path, "r") as files:
      cv={
          'name':path.name,
           'skills':find_skills(''.join(files.readlines()))
      }
      cvs.append(cv)

In [ ]:
print(cvs[1]['name'])
print(cvs[1]['skills'])
print(cvs[2]['name'])
print(cvs[2]['skills'])
print(cvs[3]['name'])
print(cvs[3]['skills'])
print(cvs[4]['name'])
print(cvs[4]['skills'])
print(cvs[5]['name'])
print(cvs[5]['skills'])

r17.txt
['Java EE', 'SQL Server', '.NET core', 'C#', 'ASP.NET', 'Rdlc', 'Linq', 'Sql', 'Web Api', 'Mvc', 'Javascript', 'Web Services', 'Oracle', 'MS SQL']
r6.txt
['J2EE', 'Oracle Fusion', 'Oracle Cloud', 'Salesforce', 'Devops Android', 'Business Analyst', 'UI Developer', 'DBAs', 'Embedded Systems', '.NET', 'Hadoop', 'SQL Developer', 'Big Data', 'Tableau', 'Networking', 'Etl', 'Informatica', 'Ios', 'Quality Analyst', 'Project Manager', 'Python']
r16.txt
['IHS', 'WAS', 'Java EE', 'SQL Server', '.NET core', 'C#', 'ASP.NET', 'Rdlc', 'Linq', 'Sql', 'Web Api', 'Mvc', 'Javascript', 'Web Services', 'Oracle', 'MS SQL']
r4.txt
['MySQL', 'PostgreSQL', 'Microsoft Access', 'SQL Server', 'FileMaker', 'Oracle']
r3.txt
['MySQL', 'PostgreSQL', 'Microsoft Access', 'SQL Server', 'FileMaker', 'Oracle', 'RDBMS', 'dBASE']


### Matching both list cv and jobs


In [ ]:
def job_match(text,cv=True):
  skills=find_skills(text)
  matched=[]
  if cv:
    for job in jobs:
      nskill_job=len(job['skills'])
      count=0
      for skill in skills:
        if skill in job['skills']:
          count+=1
      matched.append({
          'name':job['name'],
          'pct':count/nskill_job*100,
          'job_skill':job['skills'],
          'cv_skill':skills

      })
  else:
    for cv in cvs:
      nskill_cv=len(cv['skills'])
      count=0
      for skill in skills:
        if skill in cv['skills']:
          count+=1
      matched.append({
          'name':cv['name'],
          'pct':count/nskill_cv*100,
          'job_skill':cv['skills'],
          'cv_skill':skills

      })
  return matched



### Finding Most Matching Job


In [ ]:
# find most matching jobs
#######################reading the file from folder######################
f = open('/content/drive/My Drive/five_class_entitydata/cv/r1.txt', 'r')
text = f.read()
match_jobs=job_match(text)
match_jobs = sorted(match_jobs, key=lambda k: k['pct'],reverse=True)

In [ ]:
for i in range(3):
  print(f"cv matching with {match_jobs[i]['name']}")
  print(f"{match_jobs[i]['pct']}")

cv matching with backenddeveloper.txt
100.0
cv matching with javadeveloper.txt
11.76470588235294
cv matching with phpdeveloper.txt
0.0


### Finding Most Matching Resumes


In [ ]:
# find most matching cv
#######################reading the file from folder######################
f = open('/content/drive/My Drive/five_class_entitydata/jobs/dataengineer.txt', 'r')
text = f.read()
match_cvs=job_match(text,cv=False)
match_cvs = sorted(match_cvs, key=lambda k: k['pct'],reverse=True)

In [ ]:
for i in range(10):
  print(f"job matching with cv {match_cvs[i]['name']}")
  print(f"{match_cvs[i]['pct']}")

job matching with cv r7.txt
89.47368421052632
job matching with cv r8.txt
85.71428571428571
job matching with cv r6.txt
85.71428571428571
job matching with cv r9.txt
83.33333333333334
job matching with cv r10.txt
75.0
job matching with cv r13.txt
5.555555555555555
job matching with cv r12.txt
5.128205128205128
job matching with cv r11.txt
4.878048780487805
job matching with cv r15.txt
3.571428571428571
job matching with cv r14.txt
3.125


### Cleanups


In [ ]:
##################################### delete produced resume files
i=10
while i < 30:
  print ("resume"+str(i)+".txt")
  if os.path.isfile("resume"+str(i)+".txt"):
    print ("found")
    path = "resume"+str(i)+".txt"
    os.remove(path)
    print ("deleted")
    print ("..........")
  else:
    print ("not found")
  i+=1

resume10.txt
found
deleted
..........
resume11.txt
found
deleted
..........
resume12.txt
found
deleted
..........
resume13.txt
found
deleted
..........
resume14.txt
found
deleted
..........
resume15.txt
found
deleted
..........
resume16.txt
found
deleted
..........
resume17.txt
found
deleted
..........
resume18.txt
found
deleted
..........
resume19.txt
found
deleted
..........
resume20.txt
found
deleted
..........
resume21.txt
found
deleted
..........
resume22.txt
found
deleted
..........
resume23.txt
found
deleted
..........
resume24.txt
found
deleted
..........
resume25.txt
found
deleted
..........
resume26.txt
found
deleted
..........
resume27.txt
found
deleted
..........
resume28.txt
found
deleted
..........
resume29.txt
not found


In [ ]:
###################deleting the saved model#################################
#  !rm -rf model2


### xxxxx


In [ ]:
###################loading the saved model################################
output_dir='./model2'
nlp2 = spacy.load(output_dir)

In [ ]:
#######################reading the file from folder######################
f = open('/content/drive/My Drive/five_class_entitydata/feed1.txt', 'r')
text = f.read()
# text="im competent in java,c# and python"
# text=cleandata(text)

FileNotFoundError: [Errno 2] No such file or directory: '/content/drive/My Drive/five_class_entitydata/feed1.txt'

In [ ]:
docx=nlp2(text)
d = {}
for ent in docx.ents:
  d[ent.label_] = []
for ent in docx.ents:
  d[ent.label_].append(ent.text)
if 'Skills' in d:
  skills_=d['Skills']
  print(f'Dedected skills {skills_}')

Dedected skills ['J2EE', 'Oracle Fusion', 'Oracle Cloud', 'Salesforce', 'Business Analyst', 'UI Developer', 'DBAs', '.NET', 'Hadoop', 'SQL Developer', 'Big Data', 'Tableau', 'Networking', 'Etl', 'Informatica', 'Ios', 'Quality Analyst', 'Python']


In [ ]:
#########################viewving the results####################
from spacy import displacy

# Use the 'docx' object created in the previous cell (_xqx4SPJfaHW)
# which contains the processed text and detected entities.
if 'docx' in locals():
    displacy.render(docx, style='ent', jupyter=True)
else:
    print("The 'docx' object was not found. Please run the previous cell (_xqx4SPJfaHW) first.")